# Product Price Predictor — Full Training (Text + Images)
**Author: Ayush Anand**

This notebook trains **7 ML models** on both text features and image features.

### Before you start:
1. Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`
2. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/price-predictor/dataset/`
3. Upload the entire project folder (optional — only needed if using the src/ pipeline)

### Time estimates on Colab T4 GPU:
| Step | Time |
|------|------|
| Install dependencies | 3 min |
| Download 600k images | 3–5 hours |
| Extract image features (ResNet50) | 45–60 min |
| Extract text features | 10–20 min |
| Train 7 models | 60–90 min |
| **Total** | **~6–8 hours** |

## Step 1 — Install dependencies

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q xgboost lightgbm scikit-learn pandas numpy tqdm pillow requests
print('All packages installed.')

## Step 2 — Mount Google Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

# --- Change this path if your Drive folder is different ---
DRIVE_PATH = '/content/drive/MyDrive/price-predictor'

# Copy datasets to Colab local storage (faster reads)
os.makedirs('/content/dataset', exist_ok=True)
os.makedirs('/content/images', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

shutil.copy(f'{DRIVE_PATH}/dataset/train.csv', '/content/dataset/train.csv')
shutil.copy(f'{DRIVE_PATH}/dataset/test.csv',  '/content/dataset/test.csv')

import pandas as pd
train_df = pd.read_csv('/content/dataset/train.csv')
test_df  = pd.read_csv('/content/dataset/test.csv')
print(f'Train: {len(train_df):,} rows')
print(f'Test:  {len(test_df):,} rows')
train_df.head(3)

## Step 3 — Download product images

This downloads images from Amazon CDN URLs in `image_link` column.
Uses 50 parallel workers — estimated 3–5 hours for 600k images.

> **Tip:** After images are downloaded once, they are saved to Google Drive so you don't need to re-download on the next session.

In [ ]:
import requests
import concurrent.futures
import time
from tqdm.notebook import tqdm
from pathlib import Path

IMAGE_DIR = Path('/content/images')
IMAGE_DIR.mkdir(exist_ok=True)

all_rows = pd.concat([train_df, test_df], ignore_index=True)

def download_image(row):
    """Download a single image; returns (sample_id, status)."""
    sid  = str(row['sample_id'])
    url  = str(row.get('image_link', ''))
    dest = IMAGE_DIR / f'{sid}.jpg'

    if dest.exists():
        return sid, 'cached'
    if not url or url == 'nan':
        return sid, 'no_url'

    for attempt in range(3):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                dest.write_bytes(r.content)
                return sid, 'ok'
        except Exception:
            time.sleep(1)
    return sid, 'failed'

rows_list = list(all_rows.itertuples(index=False))
stats = {'ok': 0, 'cached': 0, 'failed': 0, 'no_url': 0}

print(f'Downloading {len(rows_list):,} images with 50 workers ...')
print('This will take 3-5 hours. Do not close the browser tab.\n')

with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(download_image, row): row for row in rows_list}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(rows_list)):
        _, status = future.result()
        stats[status] = stats.get(status, 0) + 1

print('\nDownload complete!')
for k, v in stats.items():
    print(f'  {k}: {v:,}')

# Save images to Drive for reuse across sessions
print('\nSyncing images to Google Drive (for future sessions)...')
!rsync -a /content/images/ {DRIVE_PATH}/images/ --ignore-existing
print('Sync done.')

## Step 4 — Extract features

### What features are extracted:

**Text features** (fast, CPU) — from `catalog_content`:
- Product value and unit (e.g. "72 Fl Oz")
- Pack quantity (e.g. "Pack of 6")
- Title word count, total text length, digit ratio
- Brand detection (Apple, Samsung, Nike, etc.)
- Category signals (electronics, food, clothing, etc.)

**Image features** (slow, GPU) — from product photos:
- 2048-dimensional vector from ResNet50 (pre-trained on ImageNet)
- These capture visual patterns: product color, shape, packaging style
- Products with premium-looking images tend to cost more — the CNN captures this

In [ ]:
import numpy as np
import re
from sklearn.preprocessing import StandardScaler

# ============================================================
# Text feature extraction
# ============================================================
def parse_catalog(content):
    text = str(content).lower()

    def extract_float(prefix):
        idx = text.find(prefix)
        if idx == -1: return 0.0
        m = re.search(r'[\d]+\.?[\d]*', text[idx+len(prefix):idx+len(prefix)+30])
        return float(m.group()) if m else 0.0

    def extract_pack():
        for pat in [r'pack\s+of\s+(\d+)', r'\((\d+)\s+pack\)', r'(\d+)\s*(?:count|pcs|pieces|ct\b)']:
            m = re.search(pat, text)
            if m: return float(m.group(1))
        return 1.0

    value    = extract_float('value:')
    pack_qty = extract_pack()
    brands   = ['apple','samsung','sony','lg','hp','dell','nike','adidas','amazon','google']
    cats     = {'electronic':1,'food':2,'clothing':3,'toy':4,'supplement':5,'cable':1,'sauce':2,'shirt':3}
    category = next((v for k,v in cats.items() if k in text), 0)

    return [
        value, pack_qty, len(content), len(content.split()),
        sum(c.isdigit() for c in content) / max(len(content),1),
        float(any(b in text for b in brands)),
        float(category),
        float(any(k in text for k in ['inch','cm','mm','size'])),
        float(any(k in text for k in ['oz','lb','gram','kg'])),
        float(any(k in text for k in ['ml','liter','gallon'])),
        value * max(pack_qty,1),
        np.log1p(value),
        np.log1p(len(content.split())),
    ]

print('Extracting text features...')
X_train_text = np.array([parse_catalog(c) for c in tqdm(train_df['catalog_content'].fillna(''))], dtype=np.float32)
X_test_text  = np.array([parse_catalog(c) for c in tqdm(test_df['catalog_content'].fillna(''))],  dtype=np.float32)

scaler = StandardScaler()
X_train_text = scaler.fit_transform(X_train_text)
X_test_text  = scaler.transform(X_test_text)
print(f'Text features: train {X_train_text.shape}, test {X_test_text.shape}')

In [ ]:
# ============================================================
# Image feature extraction using ResNet50
# ============================================================
import torch
import torchvision.models as tv_models
import torchvision.transforms as T
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Load ResNet50 — strip the final classification layer
resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = torch.nn.Identity()   # output: 2048-dim vector per image
resnet.eval().to(device)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

IMAGE_DIR = Path('/content/images')
FEAT_DIM  = 2048

def extract_cnn_features(df, batch_size=128):
    """Extract 2048-dim ResNet50 features for all rows in df."""
    features = np.zeros((len(df), FEAT_DIM), dtype=np.float32)
    n = len(df)

    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc='CNN batches'):
            batch_ids = df['sample_id'].iloc[start:start+batch_size].tolist()
            imgs, valid = [], []

            for i, sid in enumerate(batch_ids):
                path = IMAGE_DIR / f'{sid}.jpg'
                if path.exists():
                    try:
                        img = Image.open(path).convert('RGB')
                        imgs.append(transform(img))
                        valid.append(start + i)
                    except Exception:
                        pass

            if imgs:
                out = resnet(torch.stack(imgs).to(device)).cpu().numpy()
                for j, idx in enumerate(valid):
                    features[idx] = out[j]

    return features

print('Extracting image features for TRAIN set (~30-45 min on T4)...')
X_train_img = extract_cnn_features(train_df)

print('Extracting image features for TEST set...')
X_test_img  = extract_cnn_features(test_df)

# Save to Drive for reuse
np.save(f'{DRIVE_PATH}/X_train_img.npy', X_train_img)
np.save(f'{DRIVE_PATH}/X_test_img.npy',  X_test_img)
print('Image features saved to Drive.')

In [ ]:
# Combine text + image features
X_train = np.hstack([X_train_text, X_train_img])
X_test  = np.hstack([X_test_text,  X_test_img])
y_train = train_df['price'].values.astype(np.float64)

print(f'Combined train features: {X_train.shape}')
print(f'Combined test features:  {X_test.shape}')
print(f'Price range in training: {y_train.min():.2f} – {y_train.max():.2f}')

## Step 5 — Train 7 Models

| # | Model | Why we use it |
|---|-------|---------------|
| 1 | Random Forest | Robust, handles outliers well |
| 2 | Extra Trees | Faster than RF, more randomness reduces overfitting |
| 3 | XGBoost | State-of-the-art gradient boosting |
| 4 | LightGBM | Faster XGBoost variant, better on large datasets |
| 5 | Gradient Boosting | sklearn's sequential boosting, different bias |
| 6 | Ridge Regression | Fast linear baseline, good anchor for ensemble |
| 7 | Neural Network | Learns non-linear patterns in combined feature space |

All models are evaluated with **5-fold cross-validation using SMAPE** (the competition metric).  
Final predictions are a **weighted ensemble** — models with lower SMAPE get higher weight.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
import xgboost as xgb
import lightgbm as lgb
import pickle, time

SEED = 42

def smape(y_true, y_pred):
    y_pred = np.clip(y_pred, 0.01, None)
    denom  = (np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)

def cv_train(name, model, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = []
    for fold, (tr, val) in enumerate(kf.split(X)):
        model.fit(X[tr], y[tr])
        preds = np.clip(model.predict(X[val]), 0.01, None)
        s = smape(y[val], preds)
        scores.append(s)
        print(f'  {name} fold {fold+1}: SMAPE = {s:.2f}%')
    return float(np.mean(scores))

# Define all 7 models
models = {
    'random_forest':    RandomForestRegressor(n_estimators=200, max_depth=20, random_state=SEED, n_jobs=-1),
    'extra_trees':      ExtraTreesRegressor(n_estimators=200, max_depth=20, random_state=SEED, n_jobs=-1),
    'xgboost':          xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, random_state=SEED, n_jobs=-1, verbosity=0),
    'lightgbm':         lgb.LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, num_leaves=63, subsample=0.8, random_state=SEED, n_jobs=-1, verbose=-1),
    'gradient_boosting':GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=SEED),
    'ridge_regression': Ridge(alpha=10.0),
}

results    = {}
test_preds = {}

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'Training: {name.upper()}')
    print(f'{"="*50}')
    t0 = time.time()

    cv_smape = cv_train(name, model, X_train, y_train)

    # Retrain on full data
    model.fit(X_train, y_train)
    preds = np.clip(model.predict(X_test), 0.01, None)
    test_preds[name] = preds
    results[name]    = cv_smape

    with open(f'/content/models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)

    print(f'  Mean CV SMAPE: {cv_smape:.2f}%  | Time: {time.time()-t0:.0f}s')

print('\nAll models trained!')

In [ ]:
# ============================================================
# Model 7: Neural Network (PyTorch MLP)
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
INPUT_DIM = X_train.shape[1]

class PriceMLP(nn.Module):
    """Multi-layer perceptron for price regression."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.net(x)

nn_model = PriceMLP().to(device)
opt      = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
loss_fn  = nn.MSELoss()

Xt = torch.FloatTensor(X_train).to(device)
yt = torch.FloatTensor(y_train.reshape(-1, 1)).to(device)
loader = DataLoader(TensorDataset(Xt, yt), batch_size=512, shuffle=True)

print('Training Neural Network...')
EPOCHS = 50
for ep in range(EPOCHS):
    nn_model.train()
    epoch_loss = 0
    for bx, by in loader:
        opt.zero_grad()
        loss = loss_fn(nn_model(bx), by)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    if (ep+1) % 10 == 0:
        print(f'  Epoch {ep+1}/{EPOCHS}  loss={epoch_loss/len(loader):.4f}')

nn_model.eval()
with torch.no_grad():
    nn_preds = nn_model(torch.FloatTensor(X_test).to(device)).cpu().numpy().flatten()
nn_preds = np.clip(nn_preds, 0.01, None)

# Quick SMAPE on a validation slice
val_size  = int(0.1 * len(X_train))
nn_model2 = PriceMLP().to(device)
nn_model2.load_state_dict(nn_model.state_dict())
nn_model2.eval()
with torch.no_grad():
    val_preds = nn_model2(torch.FloatTensor(X_train[-val_size:]).to(device)).cpu().numpy().flatten()
nn_cv_smape = smape(y_train[-val_size:], np.clip(val_preds, 0.01, None))

test_preds['neural_network'] = nn_preds
results['neural_network']    = nn_cv_smape
print(f'  Neural Network SMAPE (holdout): {nn_cv_smape:.2f}%')

## Step 6 — Weighted Ensemble & Submit

In [ ]:
# ============================================================
# Build weighted ensemble
# Weight = 1/SMAPE  (lower SMAPE => higher weight)
# ============================================================
print('Model performance summary:')
print(f'{"Model":<25}  {"CV SMAPE":>10}')
print('-' * 38)
for name, s in sorted(results.items(), key=lambda x: x[1]):
    print(f'{name:<25}  {s:>9.2f}%')

weights   = {n: 1.0/s for n, s in results.items()}
total_w   = sum(weights.values())
weights   = {n: w/total_w for n, w in weights.items()}

print('\nEnsemble weights:')
for n, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f'  {n:<25}: {w:.3f}')

ensemble = np.zeros(len(X_test))
for n, w in weights.items():
    ensemble += w * test_preds[n]
ensemble = np.clip(ensemble, 0.01, None)

# Write submission file
submission = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': ensemble})
submission.to_csv('/content/test_out.csv', index=False)

# Also save to Drive
submission.to_csv(f'{DRIVE_PATH}/test_out.csv', index=False)

print(f'\nSubmission saved: /content/test_out.csv')
print(f'Rows: {len(submission):,}')
print(f'Price range: {ensemble.min():.2f} – {ensemble.max():.2f}')
print(f'Mean price:  {ensemble.mean():.2f}')

In [ ]:
# Download test_out.csv to your local machine
from google.colab import files
files.download('/content/test_out.csv')
print('Download started!')

---
## Notes for learning

**Why 7 models instead of 1?**  
Each model has different strengths. Random Forest handles outliers well. XGBoost/LightGBM are excellent on tabular data. Ridge provides a stable linear baseline. Combining them (ensemble) almost always beats any single model.

**Why SMAPE instead of MSE?**  
SMAPE (Symmetric Mean Absolute Percentage Error) treats errors on cheap and expensive products equally. A \$1 error on a \$5 product is worse than a \$1 error on a \$500 product — SMAPE captures this.

**Why ResNet50 for images?**  
ResNet50 is pre-trained on 1.2 million images (ImageNet). It already "knows" how to recognize shapes, textures, colors, and objects. We use it as a feature extractor (transfer learning) rather than training from scratch — this saves weeks of compute.

**What does the neural network learn that trees can't?**  
Neural networks learn smooth, continuous functions and can find complex non-linear interactions between text and image features. Trees split on individual features one at a time and miss these subtle cross-feature patterns.